In [8]:
# ==========================================================
# Generate tabular_2 (Distribution Shift + Noise)
# Based on tabular_1 dataset (6-2-2 split)
# ==========================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# ---------- Configuration ----------
SEED = 42
LABEL_COL = "Diabetes_012"      # target column name from tabular_1
N_SHIFT_FEATURES = 6            # number of numeric features to shift
SCALE_RANGE = (0.8, 1.3)        # scaling range for random rescaling
SHIFT_SIGMA_FRAC = 0.10         # mean shift ~ 0.10 * σ
NOISE_SIGMA_FRAC = 0.05         # sample-level Gaussian noise ~ 0.05 * σ
EXPLICIT_SHIFT_COLS = []        # specify columns to shift (leave empty for random selection)

np.random.seed(SEED)

# ---------- Path handling ----------
def find_datasets_dir(start: Path) -> Path:
    """Search upward from the current path to locate the 'Datasets' folder."""
    p = start.resolve()
    for _ in range(6):
        if p.name == "Datasets":
            return p
        if (p / "Datasets").exists():
            return (p / "Datasets").resolve()
        p = p.parent
    raise FileNotFoundError("Could not locate the 'Datasets' folder.")

# Locate main dataset folders automatically (compatible with Jupyter)
DATASETS_DIR = find_datasets_dir(Path.cwd())
SITE1_DIR = DATASETS_DIR / "tabular_1"
SITE2_DIR = DATASETS_DIR / "tabular_2"
SITE2_DIR.mkdir(parents=True, exist_ok=True)

# File paths
TRAIN_P = SITE1_DIR / "diabetes_012_train.csv"
VAL_P   = SITE1_DIR / "diabetes_012_val.csv"
TEST_P  = SITE1_DIR / "diabetes_012_test.csv"

print("Checking file paths:")
print("Train:", TRAIN_P.exists(), TRAIN_P)
print("Val  :", VAL_P.exists(),   VAL_P)
print("Test :", TEST_P.exists(),  TEST_P)

# ---------- Load and merge ----------
train = pd.read_csv(TRAIN_P)
val   = pd.read_csv(VAL_P)
test  = pd.read_csv(TEST_P)
df = pd.concat([train, val, test], ignore_index=True)

assert LABEL_COL in df.columns, f"'{LABEL_COL}' not found in columns: {df.columns.tolist()}"

feature_cols = [c for c in df.columns if c != LABEL_COL]
num_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

# ---------- Select columns for distribution shift ----------
if EXPLICIT_SHIFT_COLS:
    shift_cols = [c for c in EXPLICIT_SHIFT_COLS if c in num_cols]
else:
    n_pick = min(N_SHIFT_FEATURES, len(num_cols))
    rng = np.random.default_rng(SEED)
    shift_cols = rng.choice(num_cols, size=n_pick, replace=False).tolist()

print("\nTotal features:", len(feature_cols), "| numeric:", len(num_cols))
print("Selected columns for shift:", shift_cols)

# ---------- Apply distribution shift + noise ----------
def safe_std(s: pd.Series) -> float:
    std = float(s.std(ddof=0))
    return std if std > 1e-12 else 1.0

df_shifted = df.copy()
before_stats = {c: (float(df[c].mean()), float(df[c].std(ddof=0))) for c in shift_cols}

for col in shift_cols:
    sigma = safe_std(df[col])
    scale = np.random.uniform(*SCALE_RANGE)
    shift = np.random.normal(0.0, SHIFT_SIGMA_FRAC * sigma)
    noise = np.random.normal(0.0, NOISE_SIGMA_FRAC * sigma, len(df))
    df_shifted[col] = df[col] * scale + shift + noise

after_stats = {c: (float(df_shifted[c].mean()), float(df_shifted[c].std(ddof=0))) for c in shift_cols}

# ---------- Split into train / val / test (6-2-2) ----------
df_train, df_temp = train_test_split(
    df_shifted, test_size=0.20, random_state=SEED, stratify=df_shifted[LABEL_COL]
)
val_ratio = 0.20 / (0.80)
df_val, df_test = train_test_split(
    df_temp, test_size=1 - val_ratio, random_state=SEED, stratify=df_temp[LABEL_COL]
)

# ---------- Save shifted dataset ----------
OUT_TRAIN = SITE2_DIR / "diabetes_012_train_shift.csv"
OUT_VAL   = SITE2_DIR / "diabetes_012_val_shift.csv"
OUT_TEST  = SITE2_DIR / "diabetes_012_test_shift.csv"

df_train.to_csv(OUT_TRAIN, index=False)
df_val.to_csv(OUT_VAL, index=False)
df_test.to_csv(OUT_TEST, index=False)

print("\n✅ Shifted dataset successfully saved in:", SITE2_DIR)
print(" -", OUT_TRAIN.name, "shape =", df_train.shape)
print(" -", OUT_VAL.name,   "shape =", df_val.shape)
print(" -", OUT_TEST.name,  "shape =", df_test.shape)

# ---------- Label distribution check ----------
def label_dist(d):
    return d[LABEL_COL].value_counts(normalize=True).round(3).to_dict()

print("\nLabel distribution:")
print("train:", label_dist(df_train))
print("val  :", label_dist(df_val))
print("test :", label_dist(df_test))

# ---------- Summary comparison ----------
summary = []
for c in shift_cols:
    mu0, sd0 = before_stats[c]
    mu1, sd1 = after_stats[c]
    summary.append({
        "Feature": c,
        "Mean (tabular_1)": round(mu0, 4),
        "Mean (tabular_2)": round(mu1, 4),
        "Std (tabular_1)": round(sd0, 4),
        "Std (tabular_2)": round(sd1, 4),
        "ΔMean": round(mu1 - mu0, 4),
        "ΔStd": round(sd1 - sd0, 4),
        "Scale": f"{SCALE_RANGE[0]}~{SCALE_RANGE[1]}",
        "Shift": f"N(0,{SHIFT_SIGMA_FRAC}σ)",
        "Noise": f"N(0,{NOISE_SIGMA_FRAC}σ)"
    })

summary_df = pd.DataFrame(summary, columns=[
    "Feature","Mean (tabular_1)","Mean (tabular_2)",
    "Std (tabular_1)","Std (tabular_2)","ΔMean","ΔStd","Scale","Shift","Noise"
])
print("\n=== Distribution Shift Summary ===")
print(summary_df.to_string(index=False))


Checking file paths:
Train: True /Users/xushengzhe/Desktop/5703-Federated-Model/Datasets/tabular_1/diabetes_012_train.csv
Val  : True /Users/xushengzhe/Desktop/5703-Federated-Model/Datasets/tabular_1/diabetes_012_val.csv
Test : True /Users/xushengzhe/Desktop/5703-Federated-Model/Datasets/tabular_1/diabetes_012_test.csv

Total features: 21 | numeric: 21
Selected columns for shift: ['NoDocbcCost', 'Sex', 'MentHlth', 'BMI', 'Veggies', 'HighChol']

✅ Shifted dataset successfully saved in: /Users/xushengzhe/Desktop/5703-Federated-Model/Datasets/tabular_2
 - diabetes_012_train_shift.csv shape = (183769, 22)
 - diabetes_012_val_shift.csv shape = (11485, 22)
 - diabetes_012_test_shift.csv shape = (34458, 22)

Label distribution:
train: {0.0: 0.827, 1.0: 0.173}
val  : {0.0: 0.827, 1.0: 0.173}
test : {0.0: 0.827, 1.0: 0.173}

=== Distribution Shift Summary ===
    Feature  Mean (tabular_1)  Mean (tabular_2)  Std (tabular_1)  Std (tabular_2)   ΔMean    ΔStd   Scale     Shift      Noise
NoDocbcCos